# ARGUS — PC Test Phase (Option C)

**Two-Speed / YOLO-World architecture — local validation on this PC, before any Jetson work.**

Runs **entirely locally** on a VS Code Jupyter kernel (base conda env, GTX 1060 6GB). **No Colab.** It does **NOT** run Gemma 4 / any VLM — the reasoning brain is deferred to the Jetson (see the pipeline doc, §3 & §6.4).

What we validate here (the *fast path* perception + speech I/O):
1. All three cameras (2× AR0234 stereo + 1× IMX477P wide) open and stream
2. Stereo pair (AR0234) is time-synchronized
3. **YOLO-World** open-vocabulary detection on the wide camera (the grounding tool)
4. Speech stack: openWakeWord + faster-whisper (tiny, INT8) + Piper
5. ONNX export of YOLO-World (`dynamo=False`, opset 17) — ready for Jetson TensorRT

> **GTX 1060 (Pascal) note:** 6 GB VRAM, compute capability 6.1, **no Tensor Cores** → use **FP32** locally (FP16/AMP gives no speedup on Pascal). The official plan fine-tunes with `amp=True` on Colab T4/L4; locally we keep FP32. Plenty for YOLO-World inference + light tuning; speech runs on CPU.

> **What is deferred to the Jetson (do NOT run here):** Gemma 4 E2B (VLM, via llama.cpp), all TensorRT engine builds, the full two-speed runtime.

## Cell 1 — Environment detection & setup

**What it does:** Detects local-vs-Colab (we expect *local*), sets `BASE = D:\ARGUS`, builds the folder tree, and confirms the 1060 is visible to PyTorch with CUDA.

**Success looks like:** `Environment: LOCAL`, `CUDA available: True`, `GPU: NVIDIA GeForce GTX 1060 6GB`, compute cap `6.1`. All folders created under `D:\ARGUS`.

In [ ]:
import os, sys, platform
from pathlib import Path

# --- Detect environment (we expect LOCAL on this PC) ---
try:
    import google.colab  # noqa
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    BASE = Path('/content/ARGUS')  # fallback only — this project is meant to run LOCALLY
    print('WARNING: Running on Colab. This notebook is designed for LOCAL PC use (cameras are USB-attached).')
else:
    BASE = Path(r'D:\ARGUS')

print(f'Environment: {"COLAB" if IS_COLAB else "LOCAL"}')
print(f'Python     : {platform.python_version()}  ({sys.executable})')
print(f'BASE       : {BASE}')

# --- Create folder tree (artefacts only; datasets staged locally per efficiency rules) ---
for sub in ['datasets', 'models', 'models/piper', 'exports/onnx', 'logs', 'calibration']:
    (BASE / sub).mkdir(parents=True, exist_ok=True)
print('Folder tree ready under', BASE)

# --- GPU / CUDA check ---
try:
    import torch
    print(f'\ntorch       : {torch.__version__}')
    print(f'CUDA avail  : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU         : {torch.cuda.get_device_name(0)}')
        cap = torch.cuda.get_device_capability(0)
        print(f'Compute cap : {cap[0]}.{cap[1]}  (Pascal=6.1 -> FP32 only, no Tensor Cores)')
        print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    else:
        print('CUDA NOT available — install the CUDA build of torch (see SETUP_LOCAL.md).')
except ImportError:
    print('torch not installed yet — run Cell 2, then re-run this cell.')

## Cell 2 — Dependency check / install

**What it does:** Verifies (and installs if missing) the packages for the PC test, prints versions.

**Success looks like:** every package reports a version, no FAIL. `onnxruntime` (CPU) is only used to validate the export; YOLO inference uses torch on the 1060 (FP32).

> `torch`/`torchvision` are **not** auto-installed here — they must be the CUDA build matching the 1060. Install them per `SETUP_LOCAL.md` *before* running this notebook, or Cell 1 will report `CUDA avail: False`.

In [ ]:
import importlib, subprocess, sys

# pip-name -> import-name
REQUIRED = {
    'ultralytics':    'ultralytics',     # YOLO-World (grounding tool)
    'opencv-python':  'cv2',             # cameras
    'faster-whisper': 'faster_whisper',  # STT (CTranslate2, INT8 on CPU)
    'openwakeword':   'openwakeword',    # wake word
    'onnx':           'onnx',            # export
    'onnxruntime':    'onnxruntime',     # export validation (CPU)
    'sounddevice':    'sounddevice',     # mic capture / playback (CPU)
    'soundfile':      'soundfile',       # wav read for playback
    'piper-tts':      'piper',           # TTS
}

def ensure(pip_name, import_name):
    try:
        m = importlib.import_module(import_name)
        print(f'  OK   {pip_name:<16} {getattr(m, "__version__", "?")}')
    except ImportError:
        print(f'  ...  installing {pip_name}')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=False)
        try:
            m = importlib.import_module(import_name)
            print(f'  OK   {pip_name:<16} {getattr(m, "__version__", "?")} (installed)')
        except ImportError as e:
            print(f'  FAIL {pip_name}: {e}')

print('Dependency check:')
for pip_name, import_name in REQUIRED.items():
    ensure(pip_name, import_name)

print('\nNote: install CUDA-enabled torch separately (SETUP_LOCAL.md). FP32 only on Pascal 1060.')

## Cell 3 — Camera enumeration

**What it does:** Probes OpenCV `VideoCapture` indices 0–7, reports which open, and prints native resolution + FPS for each. Use this to map indices → physical cameras.

**Success looks like:** you see **3** working indices. The two with identical AR0234 resolution (typically 1920×1200) are the stereo pair; the high-res one (up to 4056×3040) is the IMX477P wide.

> **Windows:** we use the `CAP_DSHOW` backend — it enumerates UVC cameras more reliably than the default MSMF and avoids long open delays.

In [ ]:
import cv2

MAX_INDEX = 8
found = []
print(f'Probing camera indices 0..{MAX_INDEX-1} (CAP_DSHOW)...')
for idx in range(MAX_INDEX):
    cap = cv2.VideoCapture(idx, cv2.CAP_DSHOW)
    if cap.isOpened():
        ok, frame = cap.read()
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        print(f'  index {idx}: OPEN  {w}x{h} @ {fps:.0f}fps  read_ok={ok}  frame={frame.shape if ok else None}')
        found.append({'index': idx, 'w': w, 'h': h, 'fps': fps})
    cap.release()

print(f'\n{len(found)} camera(s) detected: indices {[c["index"] for c in found]}')
print('Map them: two matching-resolution cams = AR0234 stereo pair; highest-res cam = IMX477P wide.')
print('Set LEFT_IDX / RIGHT_IDX / WIDE_IDX in Cell 4 and Cell 5 from what you see here.')

## Cell 4 — Three-camera live test + stereo sync check

**What it does:** Opens all three cameras, saves a sample JPEG per camera to `D:\ARGUS\logs`, and for the AR0234 pair captures a left/right pair measuring the **timestamp delta** to confirm synchronization.

**Success looks like:** three sample images saved; stereo `|delta|` reported in ms. For software sync over USB, **< ~15 ms** is acceptable for validation (true hardware-triggered sync on the Jetson would be sub-ms).

> **Set your indices first** from Cell 3. `grab()`→`retrieve()` minimizes the inter-camera capture gap.
> **Bandwidth:** three USB3 cameras can saturate one controller. If a camera fails to grab, move cameras to separate USB3 buses (see SETUP_LOCAL.md).

In [ ]:
import cv2, time
from pathlib import Path

# >>> SET THESE from Cell 3 output <<<
LEFT_IDX  = 0   # AR0234 left
RIGHT_IDX = 1   # AR0234 right
WIDE_IDX  = 2   # IMX477P wide

LOG = Path(r'D:\ARGUS\logs'); LOG.mkdir(parents=True, exist_ok=True)

def open_cam(idx):
    c = cv2.VideoCapture(idx, cv2.CAP_DSHOW)
    return c if c.isOpened() else None

cams = {'left': open_cam(LEFT_IDX), 'right': open_cam(RIGHT_IDX), 'wide': open_cam(WIDE_IDX)}
for name, c in cams.items():
    print(f'  {name:<5}: {"OPEN" if c else "FAILED"}')

# --- Save one sample frame per camera ---
for name, c in cams.items():
    if c is None:
        continue
    for _ in range(5):
        c.read()  # warm up (discard first frames)
    ok, frame = c.read()
    if ok:
        p = LOG / f'sample_{name}.jpg'
        cv2.imwrite(str(p), frame)
        print(f'  saved {p}  ({frame.shape[1]}x{frame.shape[0]})')

# --- Stereo sync test (AR0234 pair) ---
if cams['left'] and cams['right']:
    deltas = []
    for _ in range(10):
        cams['left'].grab();  t_l = time.perf_counter()
        cams['right'].grab(); t_r = time.perf_counter()
        deltas.append((t_r - t_l) * 1000.0)
    okL, fL = cams['left'].retrieve()
    okR, fR = cams['right'].retrieve()
    if okL and okR:
        cv2.imwrite(str(LOG / 'stereo_left.jpg'),  fL)
        cv2.imwrite(str(LOG / 'stereo_right.jpg'), fR)
    avg = sum(deltas) / len(deltas)
    print(f'\nStereo grab delta: avg |{avg:.2f}| ms over {len(deltas)} pairs')
    print('  < 15 ms: good (software sync). Hardware trigger gives sub-ms on the Jetson.')
else:
    print('\nStereo pair not both open — fix indices / USB bandwidth before sync test.')

for c in cams.values():
    if c: c.release()
print('\nCameras released. Check D:\\ARGUS\\logs for sample images.')

## Cell 5 — YOLO-World open-vocabulary detection (the grounding tool)

**What it does:** Loads `yolov8s-worldv2.pt` via Ultralytics' `YOLOWorld`, lets you set **arbitrary class names at runtime** (the core of YOLO-World — no retraining), runs detection on a frame from the wide IMX477P camera, prints/draws boxes + labels + confidence. Annotated image saved to logs.

**Success looks like:** model downloads on first run, your named classes are set, objects in view are detected. On the 1060 (FP32) expect a few hundred ms per frame — fine for validation (the Jetson hits ~160 ms under TensorRT).

> Set `CLASSES` to anything. `USE_LIVE_CAMERA = False` falls back to the saved `sample_wide.jpg` from Cell 4.

In [ ]:
import cv2, shutil, torch
from pathlib import Path
from ultralytics import YOLOWorld

BASE = Path(r'D:\ARGUS'); LOG = BASE / 'logs'; MODELS = BASE / 'models'

# >>> YOUR open-vocabulary classes — name anything (the user can ask for any object) <<<
CLASSES = ['keys', 'coffee mug', 'bottle', 'door handle', 'white cane', 'chair']
WIDE_IDX = 2            # IMX477P (match Cell 3/4)
USE_LIVE_CAMERA = True
CONF = 0.25

device = 0 if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  (FP32 on Pascal 1060)')

model = YOLOWorld('yolov8s-worldv2.pt')   # pretrained, no training needed to start
model.set_classes(CLASSES)
print('Classes set:', CLASSES)

# Get a frame
if USE_LIVE_CAMERA:
    cap = cv2.VideoCapture(WIDE_IDX, cv2.CAP_DSHOW)
    for _ in range(5): cap.read()
    ok, frame = cap.read(); cap.release()
    if not ok:
        raise RuntimeError('Could not read wide camera — check WIDE_IDX / USB.')
else:
    sample = LOG / 'sample_wide.jpg'
    frame = cv2.imread(str(sample))
    if frame is None:
        raise RuntimeError(f'No saved frame at {sample} — run Cell 4 or set USE_LIVE_CAMERA=True.')

# Detect
results = model.predict(frame, conf=CONF, device=device, verbose=False)
r = results[0]
print(f'\nDetections: {len(r.boxes)}')
for b in r.boxes:
    cls = r.names[int(b.cls)]
    xyxy = [round(v) for v in b.xyxy[0].tolist()]
    print(f'  {cls:<14} conf={float(b.conf):.2f}  box={xyxy}')

out = LOG / 'yoloworld_detection.jpg'
cv2.imwrite(str(out), r.plot())
print(f'\nAnnotated image saved: {out}')

# Keep a local copy of the weights for export / reproducibility
src = Path('yolov8s-worldv2.pt')
if src.exists():
    shutil.copy(src, MODELS / 'yolov8s-worldv2.pt')
    print('Weights copied to', MODELS / 'yolov8s-worldv2.pt')

## Cell 6 — (Optional) YOLO-World fine-tune — DISABLED by default

**What it does:** *When `RUN_FINETUNE = True`*, lightly fine-tunes YOLO-World on a small **local** indoor-objects dataset, sized for 6 GB: tiny batch + gradient accumulation (`nbs`), **AMP off** (Pascal → FP32), `patience=5` early stop, periodic checkpoints to `D:\ARGUS\logs`.

**Success looks like (when enabled):** training starts without OOM, checkpoints appear in logs, early-stops when val plateaus. **Leave it `False`** for the PC test — open-vocab detection usually needs no training.

> **6 GB reality:** `imgsz=640`, `batch=4` (or 2). The official plan uses `amp=True` on Colab T4/L4; here we force `amp=False` because FP16 gives no speedup on Pascal and can hurt stability. **Data must be on local disk** (never a network/Drive mount).

In [ ]:
RUN_FINETUNE = False   # <-- keep False for PC test phase

if RUN_FINETUNE:
    from ultralytics import YOLOWorld
    from pathlib import Path

    BASE = Path(r'D:\ARGUS')
    DATA_YAML = BASE / 'datasets' / 'indoor' / 'data.yaml'  # local dataset only

    model = YOLOWorld('yolov8s-worldv2.pt')
    model.train(
        data=str(DATA_YAML),
        epochs=40,
        imgsz=640,
        batch=4,               # small for 6 GB
        nbs=64,                # nominal batch -> gradient accumulation (64/4 = 16 steps)
        amp=False,             # Pascal 1060: FP32 only, no Tensor Cores
        patience=5,            # early stopping
        save_period=10,        # periodic checkpoints
        project=str(BASE / 'logs'),
        name='yoloworld_indoor',
        device=0,
        workers=4,             # data prefetch
        cache=False,           # 6 GB: don't aggressively cache images in RAM
    )
    print('Fine-tune complete. Checkpoints in D:\\ARGUS\\logs\\yoloworld_indoor')
else:
    print('RUN_FINETUNE is False — skipping training (expected for PC test phase).')

## Cell 7 — ONNX export (for later Jetson TensorRT)

**What it does:** Exports YOLO-World to ONNX with `dynamo=False`, `opset=17`, `simplify=True`, saved to `D:\ARGUS\exports\onnx\`. Validates the graph loads in onnxruntime.

**Success looks like:** an `.onnx` appears in `exports/onnx`, and onnxruntime prints its input/output names.

> **`dynamo=False` is required** — the PyTorch 2.x dynamo ONNX path fails on certain vision ops (Phase-0 rule). **TensorRT engines are NOT built here** — that happens on the Jetson (`trtexec --fp16`), because TRT engines are device-specific.

In [ ]:
from pathlib import Path
import shutil, onnxruntime as ort
from ultralytics import YOLOWorld

BASE = Path(r'D:\ARGUS'); ONNX_DIR = BASE / 'exports' / 'onnx'; ONNX_DIR.mkdir(parents=True, exist_ok=True)
CLASSES = ['keys', 'coffee mug', 'bottle', 'door handle', 'white cane', 'chair']  # baked into the exported graph

weights = BASE / 'models' / 'yolov8s-worldv2.pt'
model = YOLOWorld(str(weights)) if weights.exists() else YOLOWorld('yolov8s-worldv2.pt')
model.set_classes(CLASSES)

# Export — Ultralytics calls torch.onnx.export under the hood. opset 17, no dynamo.
exported = model.export(format='onnx', imgsz=640, opset=17, simplify=True, dynamo=False)
print('Exported:', exported)

dst = ONNX_DIR / 'yoloworld_640.onnx'
shutil.move(str(Path(exported)), str(dst))
print('Saved to:', dst)

# Validate it loads
sess = ort.InferenceSession(str(dst), providers=['CPUExecutionProvider'])
print('\nONNX validated. Inputs :', [(i.name, i.shape) for i in sess.get_inputs()])
print('ONNX validated. Outputs:', [(o.name, o.shape) for o in sess.get_outputs()])
print('\nNext (on the Jetson): trtexec --onnx=yoloworld_640.onnx --saveEngine=yoloworld_640_fp16.engine --fp16')

## Cell 8 — Speech stack test (CPU)

**What it does:** Exercises the three speech components, **all on CPU** (no GPU):
- **openWakeWord** — load a wake model and score a buffer
- **faster-whisper (tiny, INT8)** — record a few seconds from the mic and transcribe
- **Piper TTS** — synthesize a sentence and play it

**Success looks like:** wake model loads & scores; your spoken words come back as text; you hear Piper speak.

> **Audio is CPU-only** — the 1060 is irrelevant here (matches the Jetson, where speech runs on CPU). If headless / no mic, set `RECORD_MIC = False` to test model loading + TTS only.

In [ ]:
from pathlib import Path
import numpy as np

BASE = Path(r'D:\ARGUS'); LOG = BASE / 'logs'
RECORD_MIC = True       # set False if no microphone
SAMPLE_RATE = 16000

# --- 1) openWakeWord ---
print('=== openWakeWord ===')
try:
    from openwakeword.model import Model as WakeModel
    try:
        import openwakeword
        openwakeword.utils.download_models()
    except Exception as e:
        print('  (model download note:', e, ')')
    ww = WakeModel()
    scores = ww.predict(np.zeros(SAMPLE_RATE, dtype=np.int16))  # 1s silence
    print('  Loaded. Wake models:', list(scores.keys())[:5])
except Exception as e:
    print('  openWakeWord error:', e)

# --- 2) faster-whisper (tiny, INT8 on CPU) ---
print('\n=== faster-whisper (tiny, INT8) ===')
try:
    from faster_whisper import WhisperModel
    stt = WhisperModel('tiny', device='cpu', compute_type='int8')
    print('  Model loaded (tiny/int8/cpu).')
    if RECORD_MIC:
        import sounddevice as sd
        secs = 4
        print(f'  Recording {secs}s — speak now...')
        audio = sd.rec(int(secs*SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
        sd.wait()
        segments, _ = stt.transcribe(audio.flatten(), language='en')
        print('  Transcript:', repr(' '.join(s.text for s in segments).strip()))
    else:
        print('  RECORD_MIC=False — skipped live transcription.')
except Exception as e:
    print('  faster-whisper error:', e)

# --- 3) Piper TTS ---
print('\n=== Piper TTS ===')
try:
    voice_dir = BASE / 'models' / 'piper'; voice_dir.mkdir(parents=True, exist_ok=True)
    voice = voice_dir / 'en_US-lessac-medium.onnx'
    if not voice.exists():
        print('  Piper voice not found. Download en_US-lessac-medium.onnx (+ .json) from:')
        print('    https://huggingface.co/rhasspy/piper-voices/tree/main/en/en_US/lessac/medium')
        print('  and place both files in', voice_dir)
    else:
        import wave, sounddevice as sd, soundfile as sf
        from piper import PiperVoice
        pv = PiperVoice.load(str(voice))
        wav_path = LOG / 'piper_test.wav'
        with wave.open(str(wav_path), 'wb') as wf:
            pv.synthesize('ARGUS speech test successful.', wf)
        print('  Synthesized ->', wav_path)
        data, sr = sf.read(str(wav_path), dtype='float32')
        sd.play(data, sr); sd.wait()
        print('  Played back.')
except Exception as e:
    print('  Piper error:', e)

print('\nSpeech stack test done (all CPU).')

## Cell 9 — Manifest

**What it does:** Writes `D:\ARGUS\models\pc_test_manifest.json` listing every artifact produced during this PC test and where each will live on the Jetson.

**Success looks like:** a JSON file you can hand to the Jetson-deployment step, recording exactly what was validated on the PC and what is deferred to the device.

In [ ]:
import json, platform, datetime
from pathlib import Path

BASE = Path(r'D:\ARGUS')
def exists(p): return (BASE / p).exists()

manifest = {
    'project': 'ARGUS',
    'phase': 'PC test (Option C) — Two-Speed / YOLO-World',
    'generated': datetime.datetime.now().isoformat(timespec='seconds'),
    'host': {
        'os': platform.platform(),
        'python': platform.python_version(),
        'gpu': 'NVIDIA GeForce GTX 1060 6GB (Pascal, FP32 only)',
    },
    'artifacts': {
        'yoloworld_weights': {
            'path': str(BASE / 'models' / 'yolov8s-worldv2.pt'),
            'present': exists('models/yolov8s-worldv2.pt'),
            'jetson_dest': '/opt/argus/models/yolov8s-worldv2.pt',
            'role': 'Fast-path open-vocabulary grounding tool',
        },
        'yoloworld_onnx': {
            'path': str(BASE / 'exports' / 'onnx' / 'yoloworld_640.onnx'),
            'present': exists('exports/onnx/yoloworld_640.onnx'),
            'jetson_dest': '/opt/argus/exports/yoloworld_640.onnx',
            'jetson_action': 'trtexec --onnx=yoloworld_640.onnx --saveEngine=yoloworld_640_fp16.engine --fp16',
        },
        'piper_voice': {
            'path': str(BASE / 'models' / 'piper' / 'en_US-lessac-medium.onnx'),
            'present': exists('models/piper/en_US-lessac-medium.onnx'),
            'jetson_dest': '/opt/argus/models/piper/',
            'role': 'TTS (CPU)',
        },
        'sample_frames': {
            'path': str(BASE / 'logs'),
            'role': 'Camera validation images (left/right/wide + detection)',
        },
    },
    'deferred_to_jetson': {
        'reasoning_agent': 'Gemma 4 E2B (multimodal) via native llama.cpp — NOT run on PC',
        'tensorrt': 'All TRT engine builds happen on-device (device-specific)',
        'stereo_depth': 'RAFT-Stereo / SGBM runtime + INT8 TRT engine on Jetson',
        'privacy_gate': 'SCRFD + CRAFT face/text blur (mandatory before agent sees a frame)',
        'slam': 'OpenVINS / ORB-SLAM3 native code on Jetson',
    },
    'validated_on_pc': {
        'cameras_3': None,        # fill in True/False after running
        'stereo_sync': None,
        'yoloworld_detection': None,
        'speech_stack': None,
        'onnx_export': None,
    },
}

out = BASE / 'models' / 'pc_test_manifest.json'
out.write_text(json.dumps(manifest, indent=2))
print('Manifest written:', out)
print(json.dumps(manifest, indent=2))